[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/personal-health-agent/binf4070-2026/blob/main/week-10/lab-10.ipynb)

# Week 10 Lab: Tool Routing and a Defensive Health-Agent Controller

**Course:** BINF 4070 — The Future of Personal Health Assistant

**Week:** 10

A language model can ask for a tool, but application code decides whether that
request runs. In this lab you run a small health agent whose controller sits
between the model and three read-only tools. You try to get bad requests past
the controller, write one tool yourself, and test how a tool's description
changes which tool gets picked.

By the end, you will be able to:

- tell the difference between a tool request from the model and a tool call
  that actually ran;
- name the checks a controller runs before and after a tool executes;
- write a tool function and run it through the controller;
- judge whether the tool a router picks has the data to answer the question; and
- find a date-window bug in an analysis tool (take-home).

## Run the lab

1. Open the badge and save a copy. Run the cells in order, from top to bottom.
2. Edit only the cells marked **TODO**. In class there are three, one in each part.
3. Some code cells are collapsed. Run them; you do not need to read the code.
4. The lab runs without an API key. If you have the course key, save it as
   `OPENAI_API_KEY` in Colab Secrets and turn on notebook access. The cells
   labeled "real model" then call the model; without a key they show a recorded
   run. Those calls send the question, the tool descriptions, and made-up
   results, and nothing else from the notebook.
5. If a checkpoint fails, see Troubleshooting near the end of the notebook.
6. The take-home starts after the in-class submission section.

All data in this notebook is made up. Do not type real health data into a question.

## The next 60 minutes

Work through the notebook at your own pace. A rough plan:

| Minutes | Work |
|---|---|
| 0–5 | Setup |
| 5–25 | Part 1: run the agent, then try two requests the controller should reject |
| 25–40 | Part 2: write the heart-rate tool |
| 40–55 | Part 3: label four routing questions and compare two routers |
| 55–60 | Download and submit your notebook |

Parts 1–3 count for attendance. Submit what you have at the end of class, even
if a checkpoint has not passed. Anything marked optional can wait.

## Setup

Run the two cells below. The second one prints whether the real-model cells are
on. With a key they make at most 40 model calls per runtime; without one they
show a recorded run.

In [ ]:
%pip install -q "openai>=2,<3" "pydantic>=2,<3" "pandas>=2.2,<3"

# The tools and controller use pandas and Pydantic. OpenAI is imported only when
# a key is available for the real-model cells.

In [ ]:
import json
import math
import os
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator
from datetime import datetime, timedelta, timezone

# Turn this off to stay offline even when a key is available.
USE_REAL_MODEL = True  # @param {type:"boolean"}
OPENAI_MODEL = "gpt-5.6-luna"
RUN_EXTRA_ROUTING = False  # optional: compare all eight questions instead of four
MAX_REAL_CALLS = 40            # guards against an accidental loop; resets with the runtime
MAX_ROUTE_OUTPUT_TOKENS = 60   # enough for one tool request

api_key = os.environ.get("OPENAI_API_KEY")
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if api_key is None and userdata is not None:
    try:
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        api_key = None  # missing Colab Secret is the expected key-free case

if USE_REAL_MODEL and api_key:
    from openai import OpenAI
    client = OpenAI(api_key=api_key, timeout=30.0, max_retries=0)
    print(f"Real-model cells are on ({OPENAI_MODEL}, at most {MAX_REAL_CALLS} calls).")
else:
    client = None
    reason = "USE_REAL_MODEL is off" if not USE_REAL_MODEL else "OPENAI_API_KEY is absent"
    print(f"OFFLINE MODE — {reason}. Every exercise still runs.")
    print("Real-model cells will show a recorded run.")

REAL_CLIENT = client
REAL_CALLS_USED = 0

def _spend_real_call():
    """Count one real model call. Test doubles swapped into `client` do not count."""
    global REAL_CALLS_USED
    if client is REAL_CLIENT:
        if REAL_CALLS_USED >= MAX_REAL_CALLS:
            raise RuntimeError(f"Reached the limit of {MAX_REAL_CALLS} model calls for this runtime.")
        REAL_CALLS_USED += 1

### The data

Fourteen days of made-up wearable records. Each record holds one metric for one
day, with an ID, a type, a value, and a unit. The layout follows Google Health
API v4, but nothing here is copied from a real API response. The collapsed cell
turns the records into the table `wearable_data`, one row per day, and the tools
read that table.

In [ ]:
# Hardcoded source values -> identical on every machine, no seed needed.
dates = [datetime(2026, 3, 22) + timedelta(days=i) for i in range(14)]
_daily_values = {
    "steps":             [8847, 4592, 12340, 6781, 3210, 11502, 9876, 7423, 5100, 13250, 4010, 8690, 10230, 6540],
    "resting_hr":        [62, 65, 58, 67, 71, 60, 63, 66, 69, 59, 72, 64, 61, 68],
    "calories_out":      [2340, 1850, 2780, 2100, 1620, 2650, 2410, 2050, 1780, 2890, 1690, 2280, 2560, 1950],
    "fairly_active_min": [22, 10, 38, 18, 5, 35, 28, 15, 8, 42, 6, 20, 32, 12],
    "very_active_min":   [15, 3, 32, 10, 0, 28, 18, 8, 2, 35, 1, 12, 25, 5],
    "minutes_asleep":    [420, 380, 450, 360, 310, 470, 400, 440, 350, 480, 290, 410, 460, 370],
    "sleep_efficiency":  [92, 85, 95, 82, 78, 94, 90, 93, 80, 96, 75, 88, 94, 83],
    "deep_sleep_min":    [85, 62, 105, 55, 42, 98, 78, 90, 48, 108, 40, 72, 95, 58],
    "rem_sleep_min":     [95, 70, 110, 65, 55, 105, 88, 98, 60, 115, 50, 82, 100, 68],
}
_units = {
    "steps": "count", "resting_hr": "bpm", "calories_out": "kcal",
    "fairly_active_min": "min", "very_active_min": "min",
    "minutes_asleep": "min", "sleep_efficiency": "percent",
    "deep_sleep_min": "min", "rem_sleep_min": "min",
}

GOOGLE_HEALTH_V4_FIXTURE = {
    "api_version": "v4",
    "cached_at": "2026-04-04T09:00:00Z",
    "records": [
        {"record_id": f"{metric}-{day.strftime('%Y%m%d')}",
         "data_type": metric,
         "observed_at": day.strftime("%Y-%m-%dT08:00:00Z"),
         "value": values[i], "unit": _units[metric]}
        for metric, values in _daily_values.items()
        for i, day in enumerate(dates)
    ],
}

def normalize_google_health_v4(raw: dict) -> pd.DataFrame:
    '''Validate the declared classroom envelope and pivot records by day.'''
    if raw.get("api_version") != "v4" or not isinstance(raw.get("records"), list):
        raise ValueError("invalid Google Health v4 classroom envelope")
    required = {"record_id", "data_type", "observed_at", "value", "unit"}
    rows = []
    for record in raw["records"]:
        if not required.issubset(record):
            raise ValueError("malformed Google-Health-shaped record")
        rows.append({"date": record["observed_at"][:10],
                     "metric": record["data_type"], "value": record["value"]})
    table = pd.DataFrame(rows).pivot(index="date", columns="metric", values="value")
    return table.reset_index().rename_axis(None, axis=1).sort_values("date").reset_index(drop=True)

wearable_data = normalize_google_health_v4(GOOGLE_HEALTH_V4_FIXTURE)
print("Raw v4-shaped record:", GOOGLE_HEALTH_V4_FIXTURE["records"][0])
print("\nNormalized 14-day tool table:")
print(wearable_data[["date", "steps", "resting_hr", "minutes_asleep", "sleep_efficiency"]].to_string(index=False))
print(f"\nDate range: {wearable_data['date'].iloc[0]} to {wearable_data['date'].iloc[-1]}")

### Prepared model replies

The agent in Parts 1–3 uses prepared model replies, so everyone sees the same
output and no key is needed. It recognizes only the lab's example questions.
Cells labeled "real model" call the model when you have a key.

In [ ]:
import re

FALLBACK_RESPONSES = {
    # Part 1: step count
    "steps on March 25": {
        "tool_calls": [{"name": "get_step_count", "args": {"date": "2026-03-25"}}],
        "answer": "The synthetic record contains 6,781 steps for March 25, 2026."},
    "steps did I take yesterday": {
        "tool_calls": [{"name": "get_step_count", "args": {"date": "2026-04-03"}}],
        "answer": "The synthetic record contains 10,230 steps for yesterday, April 3."},
    "walk a lot on March 30": {
        "tool_calls": [{"name": "get_step_count", "args": {"date": "2026-03-30"}}],
        "answer": "The synthetic record contains 5,100 steps for March 30."},
    "step count on January 1": {
        "tool_calls": [{"name": "get_step_count", "args": {"date": "2026-01-01"}}],
        "answer": "I don't have data for January 1, 2026. My records cover March 22 to April 4 — try a date in that range."},
    "capital of France": {
        "tool_calls": [],
        "answer": "The capital of France is Paris. That said, I'm built to help with your health data — ask me about steps, heart rate, or sleep."},
    # Part 2: heart rate
    "heart rate on March 25": {
        "tool_calls": [{"name": "get_heart_rate_summary", "args": {"date": "2026-03-25"}}],
        "answer": "The March 25 synthetic record contains a vendor-derived resting-heart-rate estimate of 67 bpm, 2,100 calories, and 28 active minutes."},
    "heart rate on March 28": {
        "tool_calls": [{"name": "get_heart_rate_summary", "args": {"date": "2026-03-28"}}],
        "answer": "The March 28 synthetic record contains a vendor-derived resting-heart-rate estimate of 63 bpm, 2,410 calories, and 46 active minutes."},
    "active on March 27": {
        "tool_calls": [{"name": "get_heart_rate_summary", "args": {"date": "2026-03-27"}}],
        "answer": "The March 27 synthetic record contains a vendor-derived resting-heart-rate estimate of 60 bpm, 2,650 calories, and 63 active minutes."},
    # Part 3: sleep
    "sleep on March 24": {
        "tool_calls": [{"name": "get_sleep_summary", "args": {"date": "2026-03-24"}}],
        "answer": "The device-estimated March 24 record contains 7.5 hours asleep, 95% efficiency, 105 deep-sleep minutes, and 110 REM minutes."},
    "sleep on March 26": {
        "tool_calls": [{"name": "get_sleep_summary", "args": {"date": "2026-03-26"}}],
        "answer": "The device-estimated March 26 record contains 5.2 hours asleep, 78% efficiency, 42 deep-sleep minutes, and 55 REM minutes. These are wearable estimates, not a diagnosis."},
    # Multi-tool examples
    "full health summary for March 28": {
        "tool_calls": [
            {"name": "get_step_count", "args": {"date": "2026-03-28"}},
            {"name": "get_heart_rate_summary", "args": {"date": "2026-03-28"}},
            {"name": "get_sleep_summary", "args": {"date": "2026-03-28"}}],
        "answer": ("Full summary for March 28, 2026:\n"
                   "- Activity: 9,876 steps.\n"
                   "- Heart: vendor-derived resting HR 63 bpm, 2,410 calories, 46 active minutes.\n"
                   "- Sleep: device-estimated 6.7 hours at 90% efficiency (78 min deep, 88 min REM).\n"
                   "These are descriptive synthetic-record fields, not clinical conclusions.")},
    "tell me about my March 26": {
        "tool_calls": [
            {"name": "get_step_count", "args": {"date": "2026-03-26"}},
            {"name": "get_heart_rate_summary", "args": {"date": "2026-03-26"}},
            {"name": "get_sleep_summary", "args": {"date": "2026-03-26"}}],
        "answer": ("March 26, 2026:\n"
                   "- Steps: 3,210.\n"
                   "- Heart: vendor-derived resting HR 71 bpm, 1,620 calories, 5 active minutes.\n"
                   "- Sleep: device-estimated 5.2 hours at 78% efficiency (42 min deep, 55 min REM).\n"
                   "The fields co-occur on one day; they do not establish why any value changed.")},
}


def _tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def find_fallback(question):
    """Pick the demo response whose key best overlaps the question. Date tokens
    (numbers, 'yesterday') are weighted so different dates don't collide."""
    q = set(_tokens(question))
    best, best_score = None, 0
    for key, resp in FALLBACK_RESPONSES.items():
        score = sum(5 if (w.isdigit() or w == "yesterday") else 1
                    for w in _tokens(key) if w in q)
        if score > best_score:
            best, best_score = resp, score
    return best if best_score >= 3 else None


print(f"Loaded {len(FALLBACK_RESPONSES)} demo responses.")
print("Deterministic offline router ready.")

In [ ]:
# A recorded run of the real-model cells, used when no live call is possible.
RECORDED_REAL = {'model': 'gpt-5.6-luna',
 'recorded_on': '2026-11-06',
 'no_tool_reply': 'I don’t have access to your health or activity data, so I can’t '
                  'determine how many steps you took on March 25, 2026. Check your '
                  'phone’s Health/Fitness app or wearable’s activity history for that '
                  'date.',
 'raw_tool_call': {'type': 'function_call',
                   'call_id': 'call_vvKJX71trfTsjZ3QjCVZJ61h',
                   'name': 'get_step_count',
                   'arguments': '{"date":"2026-03-25"}'},
 'final_answer': 'You took **6,781 steps** on **March 25, 2026**.\n'
                 '\n'
                 '**Source:** Google Health data  \n'
                 '**Coverage:** March 22–April 4, 2026',
 'model_calls': 2,
 'routes': {'verbose': {'was I active yesterday': 'get_heart_rate_summary',
                        'how was my heart': 'get_heart_rate_summary',
                        'did I get good exercise': 'get_heart_rate_summary',
                        'my cardio numbers': 'get_heart_rate_summary',
                        'my energy and exertion': 'get_heart_rate_summary',
                        'intensity of my workouts': 'get_heart_rate_summary',
                        'how hard did I work out': 'get_heart_rate_summary',
                        'any irregular heart activity': 'get_heart_rate_summary'}}}

---

## Part 1: Run the agent, then attack it

*About 20 minutes.* You get a complete one-tool agent. Run it, then try to get a
bad request past its controller. You edit one cell in this part: the attack cell.

How the agent answers one question:
1. The application sends the question and the tool schemas to the model.
2. The model replies with a *tool call*: a function name and arguments. This is a request, not an answer.
3. The application checks the request, runs the function, and gets data back.
4. The application sends the data to the model, and the model writes the answer.

That often takes two model calls: one to request the tool and one to write the answer.

### The step-count tool

A tool is a function that returns structured data (a dict), not a sentence. The
model writes the sentence later.

In [ ]:
def get_step_count(date: str) -> dict:
    """Look up the step count for a date. Returns a dict, or an error dict if the
    date isn't in the data."""
    row = wearable_data[wearable_data["date"] == date]
    if row.empty:
        return {"error": f"No data for {date}. Available range: 2026-03-22 to 2026-04-04."}
    return {"date": date, "steps": int(row.iloc[0]["steps"]), "unit": "steps"}


print("Direct calls (no model involved yet):")
print("  March 25:", get_step_count("2026-03-25"))
print("  Jan 1   :", get_step_count("2026-01-01"))

### What the model sees

The model never sees the function's code. It sees only the name, the description,
and the arguments in the schema below. Run the cell. The schema is written in
the Chat Completions format. The real-model cells use the Responses API, so the
cell also converts the schema to that format and prints the top-level keys of both.
The description mentions Google Health; the values are still made up.

In [ ]:
# Chat Completions tool envelope (nested under `function`).
step_count_tool = {
    "type": "function",
    "function": {
        "name": "get_step_count",
        "description": (
            "Get the user's step count for a specific date from their Google Health data. "
            "Use this when the user asks about steps, walking, or daily physical "
            "activity on a particular day."
        ),
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "date": {
                    "type": "string",
                    "description": (
                        "The date to look up, YYYY-MM-DD. Resolve relative dates like "
                        "'yesterday' to an actual date. Data covers 2026-03-22 to 2026-04-04."
                    ),
                }
            },
            "required": ["date"],
        },
        "strict": True,
    },
}

def to_responses_tool(chat_tool: dict) -> dict:
    '''Convert this lab's nested Chat envelope to the flat Responses envelope.'''
    function = chat_tool["function"]
    return {
        "type": "function",
        "name": function["name"],
        "description": function["description"],
        "parameters": function["parameters"],
        "strict": function.get("strict", False),
    }


step_count_tool_responses = to_responses_tool(step_count_tool)
assert "function" in step_count_tool and "function" not in step_count_tool_responses
print("Chat Completions (nested):", list(step_count_tool))
print("Responses API (flat):", list(step_count_tool_responses))

### Predict, then ask the real model

The function works without a model. Now send the model the same question, with no
tools attached.

**Before you run the cell, predict:** what will it say?

In [ ]:
STEP_QUESTION = "How many steps did I take on March 25, 2026?"

def _show_recorded(key):
    if RECORDED_REAL is None:
        print("No recording is available yet. Add a key to run this live.")
    elif isinstance(RECORDED_REAL[key], str):
        print(f"Recorded run ({RECORDED_REAL['model']}, {RECORDED_REAL['recorded_on']}):")
        print(RECORDED_REAL[key])
    else:
        print(f"Recorded run ({RECORDED_REAL['model']}, {RECORDED_REAL['recorded_on']}):")
        print(json.dumps(RECORDED_REAL[key], indent=2))

def ask_model(tools, recorded_key):
    """One model call. Nothing in this function runs get_step_count."""
    if client is None:
        _show_recorded(recorded_key)
        return
    tool_settings = {"tools": tools, "tool_choice": "auto",
                     "parallel_tool_calls": False} if tools else {}
    try:
        _spend_real_call()
        response = client.responses.create(
            model=OPENAI_MODEL,
            input=STEP_QUESTION,
            reasoning={"effort": "none"},
            max_output_tokens=200,
            store=False,
            **tool_settings,
        )
    except Exception as exc:
        print(f"The model call failed ({type(exc).__name__}). Showing the recording instead.")
        _show_recorded(recorded_key)
        return
    calls = [item for item in response.output if getattr(item, "type", None) == "function_call"]
    if calls:
        for item in calls:
            print(json.dumps(item.model_dump(exclude_none=True), indent=2))
    else:
        print(response.output_text)

ask_model([], "no_tool_reply")

Without a tool, the model has no way to read your data. Now send the same
question with the tool schema attached.

**Before you run the cell, predict:** will this call run `get_step_count`? Will you
see the number 6,781?

In [ ]:
ask_model([step_count_tool_responses], "raw_tool_call")

### The controller

That call returned a request: a function name and arguments. Nothing ran
`get_step_count`, so the reply contains no step count. The code that decides
whether a request runs is the controller. It sits between the model and the tools.

For every request, the controller checks who is asking, whether the tool is on
the allowed list, and whether the arguments and the result have the right types.
It also stops a run that asks too many times. The model can propose an action,
but it cannot approve one.

Run the collapsed cell. You do not need to read it now; `execute_tool` holds the
checks if you want to look. The rest of the lab uses this controller unchanged.

In [ ]:
#@title Provided: the defensive controller (click to expand)
from collections import Counter
from dataclasses import dataclass
from hashlib import sha256

TODAY = "2026-04-04"   # pinned so relative dates are reproducible
MAX_ITERATIONS = 6
MAX_TOOL_CALLS = 6
MAX_RETRIES = 1  # retries after the first identical request
MAX_ARGUMENT_BYTES = 2_000
MAX_RESULT_BYTES = 20_000
MAX_OUTPUT_TOKENS = 400
OPENAI_SYNTHETIC_QUESTION = "How many steps did I take on March 25, 2026?"

SYSTEM_PROMPT = (
    f"You are a descriptive personal health assistant. Today's date is {TODAY}. "
    "Use authorized tools for measurements. Report units, source, and coverage; "
    "do not diagnose or apply generic clinical thresholds to wearable estimates."
)

class DateArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    date: str = Field(pattern=r"^\d{4}-\d{2}-\d{2}$")

    @field_validator("date")
    @classmethod
    def require_calendar_date(cls, value):
        datetime.strptime(value, "%Y-%m-%d")
        return value

class StepResult(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    date: str
    steps: int = Field(ge=0)
    unit: str = Field(pattern="^steps$")

class HeartResult(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    date: str
    resting_heart_rate_bpm: int = Field(ge=20, le=250)
    calories_burned: int = Field(ge=0)
    active_minutes: int = Field(ge=0, le=1440)
    unit: str = Field(pattern="^bpm$")

class SleepResult(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    date: str
    hours_asleep: float = Field(ge=0, le=24)
    sleep_efficiency: int = Field(ge=0, le=100)
    deep_sleep_minutes: int = Field(ge=0, le=1440)
    rem_sleep_minutes: int = Field(ge=0, le=1440)
    unit: str = Field(pattern="^hours$")

@dataclass(frozen=True)
class RequestContext:
    subject_id: str                 # authenticated server state; never model input
    permissions: frozenset[str]
    request_id: str

@dataclass(frozen=True)
class ToolSpec:
    args_type: type[BaseModel]
    result_type: type[BaseModel]
    handler: object
    permission: str
    version: str
    max_result_bytes: int = MAX_RESULT_BYTES

REQUEST_CONTEXT = RequestContext(
    subject_id="demo-user-10",
    permissions=frozenset({"wearable:read"}),
    request_id="week10-demo-request",
)
AUDIT_LOG = []
LAST_HISTORY = []

def _utc_iso():
    """Timezone-aware UTC timestamp with a compact RFC 3339 suffix."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

RESULT_TYPES = {
    "get_step_count": StepResult,
    "get_heart_rate_summary": HeartResult,
    "get_sleep_summary": SleepResult,
}

def _public_error(code, message, retryable=False):
    return {"ok": False, "error": {
        "code": code, "message": message, "retryable": retryable}}

def _audit(ctx, tool, outcome, version=None, fields=(), stop=None):
    event = {
        "request_id": ctx.request_id,
        "actor": sha256(ctx.subject_id.encode()).hexdigest()[:12],
        "tool": tool,
        "tool_version": version,
        "outcome": outcome,
        "argument_fields": sorted(fields),
        "stop": stop,
        "timestamp": _utc_iso(),
    }
    AUDIT_LOG.append(event)  # intentionally excludes health values and raw arguments

def _registry(tool_functions):
    """The supplied mapping is the allowlist; unknown names are never dispatched."""
    specs = {}
    for name, handler in tool_functions.items():
        if name not in RESULT_TYPES:
            continue
        specs[name] = ToolSpec(DateArgs, RESULT_TYPES[name], handler,
                               "wearable:read", "1.0.0")
    return specs

def _visible_schemas(tools, registry, ctx):
    allowed = {name for name, spec in registry.items()
               if spec.permission in ctx.permissions}
    return [schema for schema in tools
            if schema["function"]["name"] in allowed]

def execute_tool(call, registry, ctx=REQUEST_CONTEXT):
    """Validate one normalized call: {id, name, arguments-as-JSON-string}."""
    name = call["name"]
    spec = registry.get(name)
    if spec is None:
        _audit(ctx, name, "unknown_tool")
        return _public_error("UNKNOWN_TOOL", "That tool is not available.")
    if spec.permission not in ctx.permissions:
        _audit(ctx, name, "forbidden", spec.version)
        return _public_error("FORBIDDEN", "You are not authorized for that data.")

    argument_text = call["arguments"]
    if len(argument_text.encode("utf-8")) > MAX_ARGUMENT_BYTES:
        _audit(ctx, name, "arguments_too_large", spec.version)
        return _public_error("INVALID_ARGUMENTS", "Arguments are too large.")
    try:
        args = spec.args_type.model_validate_json(argument_text)
    except ValidationError as exc:
        fields = [".".join(map(str, err["loc"])) for err in exc.errors()]
        _audit(ctx, name, "invalid_arguments", spec.version, fields)
        return _public_error("INVALID_ARGUMENTS",
                             "Check these fields: " + ", ".join(fields))

    try:
        raw = spec.handler(**args.model_dump())
    except TimeoutError:
        _audit(ctx, name, "timeout", spec.version, args.model_fields_set)
        return _public_error("TOOL_TIMEOUT", "The data source timed out.", True)
    except Exception:
        _audit(ctx, name, "tool_failed", spec.version, args.model_fields_set)
        return _public_error("TOOL_FAILED", "The tool could not complete.")

    if isinstance(raw, dict) and "error" in raw:
        _audit(ctx, name, "no_data", spec.version, args.model_fields_set)
        return _public_error("NO_DATA", "No authorized record matched that request.")
    try:
        raw_json = json.dumps(raw, ensure_ascii=False, allow_nan=False,
                              separators=(",", ":"))
    except (TypeError, ValueError, UnicodeEncodeError):
        _audit(ctx, name, "invalid_result", spec.version)
        return _public_error("TOOL_RESULT_INVALID", "The tool returned an unusable result.")
    if len(raw_json.encode("utf-8")) > spec.max_result_bytes:
        _audit(ctx, name, "result_too_large", spec.version)
        return _public_error("TOOL_RESULT_TOO_LARGE", "The tool returned too much data.")
    try:
        typed = spec.result_type.model_validate_json(raw_json)
    except ValidationError:
        _audit(ctx, name, "invalid_result", spec.version)
        return _public_error("TOOL_RESULT_INVALID", "The tool returned an unusable result.")
    if typed.date != args.date:
        _audit(ctx, name, "result_date_mismatch", spec.version)
        return _public_error("TOOL_RESULT_INVALID", "The tool returned an unusable result.")

    _audit(ctx, name, "success", spec.version, args.model_fields_set)
    return {
        "ok": True,
        "data": typed.model_dump(mode="json"),
        "provenance": {
            "request_id": ctx.request_id,
            "subject_scope": sha256(ctx.subject_id.encode()).hexdigest()[:12],
            "source": "synthetic_google_health_shaped_fixture",
            "tool": name,
            "tool_version": spec.version,
            "retrieved_at": _utc_iso(),
        },
    }

def run_agent(user_message, tools, tool_functions, verbose=True,
              ctx=REQUEST_CONTEXT, use_openai=False):
    """Run the offline demo path, or the one fixed question through the real model."""
    global LAST_HISTORY
    registry = _registry(tool_functions)
    visible = _visible_schemas(tools, registry, ctx)
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message}]
    attempts, total_calls = Counter(), 0
    LAST_HISTORY = list(messages)

    if not use_openai:
        fb = find_fallback(user_message)
        if fb is None:
            _audit(ctx, None, "stopped", stop="no_demo_route")
            return "(offline) No listed route for that question; use a provided example."
        calls = [{"id": f"demo-call-{i+1}", "name": c["name"],
                  "arguments": json.dumps(c["args"], separators=(",", ":"))}
                 for i, c in enumerate(fb["tool_calls"])]
        if not calls:
            _audit(ctx, None, "stopped", stop="final_answer")
            return "(offline) " + fb["answer"]
        assistant_event = {"role": "assistant", "content": None,
                           "tool_calls": [{"id": c["id"], "type": "function",
                             "function": {"name": c["name"],
                                          "arguments": c["arguments"]}}
                                          for c in calls]}
        messages.append(assistant_event)  # exact request event precedes every result
        LAST_HISTORY.append(assistant_event)
        for call in calls:
            total_calls += 1
            result = execute_tool(call, registry, ctx)
            event = {"role": "tool", "tool_call_id": call["id"],
                     "content": json.dumps(result, separators=(",", ":"))}
            messages.append(event); LAST_HISTORY.append(event)
            if verbose:
                print(f"  [REQUEST] {call['id']} {call['name']}({call['arguments']})")
                print(f"  [OBSERVATION] {event['content']}")
            if not result.get("ok"):
                code = result["error"]["code"]
                stop = "policy" if code in {"FORBIDDEN", "UNKNOWN_TOOL"} else "tool_error"
                _audit(ctx, call["name"], "stopped", stop=stop)
                return result["error"]["message"]
        _audit(ctx, None, "stopped", stop="final_answer")
        return "(offline) " + fb["answer"]

    if client is None:
        raise RuntimeError(
            "The real-model run needs USE_REAL_MODEL=True and OPENAI_API_KEY."
        )
    if user_message != OPENAI_SYNTHETIC_QUESTION:
        raise ValueError(
            "This full real-model run is limited to its displayed synthetic question."
        )

    # Responses uses flat tools. Preserve each function_call item and pair it
    # with a function_call_output carrying the same call_id.
    response_input = [{"role": "user", "content": user_message}]
    responses_tools = [to_responses_tool(schema) for schema in visible]
    LAST_HISTORY = list(response_input)

    for _iteration in range(MAX_ITERATIONS):
        _spend_real_call()
        response = client.responses.create(
            model=OPENAI_MODEL,
            instructions=SYSTEM_PROMPT,
            input=response_input,
            tools=responses_tools,
            tool_choice="auto",
            parallel_tool_calls=False,
            reasoning={"effort": "none"},
            max_output_tokens=MAX_OUTPUT_TOKENS,
            store=False,
        )
        provider_items = [
            item.model_dump(exclude_none=True)
            if hasattr(item, "model_dump") else dict(item)
            for item in response.output
        ]
        response_input.extend(response.output)
        LAST_HISTORY.extend(provider_items)
        provider_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]
        if not provider_calls:
            if response.output_text:
                _audit(ctx, None, "stopped", stop="final_answer")
                return response.output_text
            _audit(ctx, None, "stopped", stop="no_valid_action")
            return "I could not determine a safe next step."

        for provider_call in provider_calls:
            total_calls += 1
            if total_calls > MAX_TOOL_CALLS:
                _audit(ctx, None, "stopped", stop="tool_budget")
                return "I stopped after too many tool requests."
            call = {"id": provider_call.call_id, "name": provider_call.name,
                    "arguments": provider_call.arguments}
            signature = (call["name"], call["arguments"])
            attempts[signature] += 1
            if attempts[signature] > 1 + MAX_RETRIES:
                _audit(ctx, call["name"], "stopped", stop="retry_budget")
                return "I stopped after the same request was attempted too many times."
            result = execute_tool(call, registry, ctx)
            event = {
                "type": "function_call_output",
                "call_id": call["id"],
                "output": json.dumps(result, separators=(",", ":")),
            }
            response_input.append(event)
            LAST_HISTORY.append(event)
            if verbose:
                print(f"  [REQUEST] {call['id']} {call['name']}({call['arguments']})")
                print(f"  [OBSERVATION] {event['output']}")
            if not result.get("ok"):
                code = result["error"]["code"]
                stop = "policy" if code in {"FORBIDDEN", "UNKNOWN_TOOL"} else "tool_error"
                _audit(ctx, call["name"], "stopped", stop=stop)
                return result["error"]["message"]
    _audit(ctx, None, "stopped", stop="iteration_budget")
    return "I could not finish within the allowed steps."

print("Defensive controller ready: allowlist + typed boundary + budgets + audit.")

### Run the agent

Look for the `[REQUEST]` and `[OBSERVATION]` lines in the output. `[REQUEST]` is
what the model asked for. `[OBSERVATION]` is the result the controller returned
after its checks passed. The collapsed cell after it shows the same run as a
list of six events and names the part of the system that handled each one.

In [ ]:
tools = [step_count_tool]
tool_functions = {"get_step_count": get_step_count}

print("Q: How many steps did I take on March 25, 2026?")
print("-" * 60)
print("Agent:", run_agent("How many steps did I take on March 25, 2026?", tools, tool_functions))

In [ ]:
# A compact request-to-observation trace from the run above. These are observable
# application events, not private model reasoning.
_assistant_request = next(e for e in LAST_HISTORY if e.get("role") == "assistant")
_tool_result = next(e for e in LAST_HISTORY if e.get("role") == "tool")
_normalized = _assistant_request["tool_calls"][0]
six_event_trace = [
    {"event": 1, "owner": "application", "record": "accept request",
     "request_id": REQUEST_CONTEXT.request_id},
    {"event": 2, "owner": "controller", "record": "apply policy",
     "eligible_tools": [t["function"]["name"] for t in tools]},
    {"event": 3, "owner": "model adapter", "record": "normalize requested action",
     "call_id": _normalized["id"], "name": _normalized["function"]["name"]},
    {"event": 4, "owner": "controller", "record": "validate and authorize",
     "decision": "allowed"},
    {"event": 5, "owner": "handler + controller", "record": "execute and validate result",
     "status": "bounded result"},
    {"event": 6, "owner": "model adapter", "record": "return paired observation then stop",
     "call_id": _tool_result["tool_call_id"]},
]
assert len(six_event_trace) == 6
assert six_event_trace[2]["call_id"] == six_event_trace[5]["call_id"]
for event in six_event_trace:
    print(event)
print("Six-event request/action/observation trace is paired.")

### Two edge cases

One question asks about a date with no data, and one is not about health. Run the
cell and check that the agent does not make up a number.

In [ ]:
for q in ["What was my step count on January 1, 2026?",   # no data for that date
          "What is the capital of France?"]:              # not a health question
    print(f"Q: {q}")
    print("Agent:", run_agent(q, tools, tool_functions))
    print("-" * 60)

### Try to get past the controller

The lecture showed where the controller checks each request. Now try to get past the real one.

`try_request` hands the controller one request that you write yourself, in place
of a model, and prints two things: the outcome, and whether the tool function
ran. The cell below defines it and sends one normal request for comparison.

In the TODO cell, write two requests that you think might slip through. Each one
is a tool name and a JSON string of arguments, like the normal request. Some
ideas: ask for another person's data by adding a `subject_id`, call a tool that
does not exist, use a date that cannot exist, or send a number where text is
expected. If one gets through, check whether it did any harm.

In [ ]:
def try_request(name, arguments, handler=get_step_count, ctx=REQUEST_CONTEXT):
    """Send one made-up request through the real controller and report the outcome."""
    runs = {"count": 0}
    def spy(date):
        runs["count"] += 1
        return handler(date)
    result = execute_tool({"id": "attack", "name": name, "arguments": arguments},
                          _registry({"get_step_count": spy}), ctx)
    outcome = "ALLOWED" if result["ok"] else result["error"]["code"]
    print(f"{name} {arguments}")
    print(f"  -> {outcome} | handler ran: {'yes' if runs['count'] else 'no'}")
    return result

_ = try_request("get_step_count", '{"date":"2026-03-25"}')   # a normal request, for comparison

In [ ]:
# TODO: Replace these two requests with your own attempts.
# Format: ("tool_name", '{"argument": "value"}')
MY_ATTACKS = [
    ("TODO_tool_name", '{"TODO": "TODO"}'),
    ("TODO_tool_name", '{"TODO": "TODO"}'),
]

for name, arguments in MY_ATTACKS:
    _ = try_request(name, arguments)


### Checkpoint: Part 1

This check sends the controller a longer list of bad requests, beyond your two.
It also tests runs that repeat the same request or make too many tool calls. No
model is called. It should print a line starting with `Part 1 checks passed`.

In [ ]:
r = get_step_count("2026-03-25")
assert r["steps"] == 6781, f"Expected 6781 steps on Mar 25, got {r['steps']}"
assert r["date"] == "2026-03-25", "The date should be echoed back"
assert get_step_count("2026-03-22")["steps"] == 8847, "Mar 22 should be 8847 steps"
assert "error" in get_step_count("2026-01-01"), "A missing date should return an error dict"

# Controller checks: allowlist, typed arguments/results, server-side identity,
# permissions, date binding, failure stopping, size caps, request/result pairing,
# behavioral budgets, and audit minimization.
_registry_step = _registry({"get_step_count": get_step_count})
_unknown = execute_tool({"id": "x", "name": "run_python", "arguments": "{}"},
                        _registry_step)
assert _unknown["error"]["code"] == "UNKNOWN_TOOL"

_oversize_arguments = execute_tool(
    {"id": "x", "name": "get_step_count",
     "arguments": json.dumps({"date": "2026-03-25",
                              "padding": "x" * MAX_ARGUMENT_BYTES})},
    _registry_step)
assert _oversize_arguments["error"]["code"] == "INVALID_ARGUMENTS"

_injected_identity = execute_tool(
    {"id": "x", "name": "get_step_count",
     "arguments": '{"date":"2026-03-25","subject_id":"other-user"}'},
    _registry_step)
assert _injected_identity["error"]["code"] == "INVALID_ARGUMENTS",     "Identity is server context and must be rejected if the model supplies it"

_invalid_calendar_date = execute_tool(
    {"id": "x", "name": "get_step_count",
     "arguments": '{"date":"2026-02-30"}'},
    _registry_step)
assert _invalid_calendar_date["error"]["code"] == "INVALID_ARGUMENTS"

_no_permission = RequestContext(subject_id="demo-user-10", permissions=frozenset(),
                                request_id="permission-negative-test")
_forbidden = execute_tool(
    {"id": "x", "name": "get_step_count",
     "arguments": '{"date":"2026-03-25"}'},
    _registry_step, ctx=_no_permission)
assert _forbidden["error"]["code"] == "FORBIDDEN"

_tiny = {"get_step_count": ToolSpec(DateArgs, StepResult, get_step_count,
                                     "wearable:read", "test", max_result_bytes=5)}
assert execute_tool({"id": "x", "name": "get_step_count",
                     "arguments": '{"date":"2026-03-25"}'}, _tiny)        ["error"]["code"] == "TOOL_RESULT_TOO_LARGE"

def _invalid_step_result(date):
    return {"date": date, "steps": -1, "unit": "steps"}

_invalid_registry = {
    "get_step_count": ToolSpec(DateArgs, StepResult, _invalid_step_result,
                               "wearable:read", "negative-test")
}
_invalid_result = execute_tool(
    {"id": "x", "name": "get_step_count",
     "arguments": '{"date":"2026-03-25"}'}, _invalid_registry)
assert _invalid_result["error"]["code"] == "TOOL_RESULT_INVALID"

# Reject values that Pydantic would otherwise coerce to integers.
for _bad_steps in ("6781", True):
    def _wrong_type_step_result(date, value=_bad_steps):
        return {"date": date, "steps": value, "unit": "steps"}
    _wrong_type = execute_tool(
        {"id": "type-test", "name": "get_step_count",
         "arguments": '{"date":"2026-03-25"}'},
        _registry({"get_step_count": _wrong_type_step_result}))
    assert _wrong_type["error"]["code"] == "TOOL_RESULT_INVALID"

def _wrong_date_step_result(date):
    return {"date": "2026-03-24", "steps": 6781, "unit": "steps"}

_wrong_date_registry = {
    "get_step_count": ToolSpec(
        DateArgs, StepResult, _wrong_date_step_result,
        "wearable:read", "wrong-date-test")
}
_wrong_date_result = execute_tool(
    {"id": "x", "name": "get_step_count",
     "arguments": '{"date":"2026-03-25"}'}, _wrong_date_registry)
assert _wrong_date_result["error"]["code"] == "TOOL_RESULT_INVALID",     "A tool result must be bound to the validated requested date"

_offline_invalid = run_agent(
    "How many steps did I take on March 25, 2026?",
    tools,
    {"get_step_count": _invalid_step_result},
    verbose=False,
)
assert _offline_invalid == "The tool returned an unusable result."
assert "6,781" not in _offline_invalid,     "A failed tool must not fall through to the precomputed answer"

_offline_no_data = run_agent(
    "What was my step count on January 1, 2026?",
    tools,
    tool_functions,
    verbose=False,
)
assert _offline_no_data == "No authorized record matched that request."
assert "March 22" not in _offline_no_data,     "A no-data result must not fall through to the precomputed answer"

run_agent("How many steps did I take on March 25, 2026?", tools,
          tool_functions, verbose=False)
_assistant_i = next(i for i, event in enumerate(LAST_HISTORY)
                    if event.get("role") == "assistant")
_tool_i = next(i for i, event in enumerate(LAST_HISTORY)
               if event.get("role") == "tool")
assert _assistant_i < _tool_i, "The exact assistant request must precede its result"
_request_id = LAST_HISTORY[_assistant_i]["tool_calls"][0]["id"]
assert LAST_HISTORY[_tool_i]["tool_call_id"] == _request_id

# Deterministic fake provider: exercise each stop condition without a key/network.

def _fake_call(call_id, date):
    return _FakeFunctionCall(
        call_id, "get_step_count",
        json.dumps({"date": date}, separators=(",", ":")))

class _FakeFunctionCall:
    type = "function_call"
    def __init__(self, call_id, name, arguments):
        self.call_id = call_id
        self.name = name
        self.arguments = arguments
    def model_dump(self, exclude_none=True):
        return {"type": self.type, "call_id": self.call_id,
                "name": self.name, "arguments": self.arguments}

class _FakeResponse:
    def __init__(self, calls, output_text=""):
        self.output = calls
        self.output_text = output_text

class _FakeResponses:
    def __init__(self, responses):
        self.responses = iter(responses)
        self.requests = []
    def create(self, **kwargs):
        self.requests.append(kwargs)
        return next(self.responses)

class _FakeClient:
    def __init__(self, responses):
        self.responses = _FakeResponses(responses)

def _run_fake_provider(responses, functions=None):
    global client
    old_client = client
    start = len(AUDIT_LOG)
    try:
        client = _FakeClient(responses)
        function_map = tool_functions if functions is None else functions
        answer = run_agent(OPENAI_SYNTHETIC_QUESTION, tools, function_map,
                           verbose=False, use_openai=True)
        return answer, AUDIT_LOG[start:], client.responses.requests
    finally:
        client = old_client

_iteration_responses = [
    _FakeResponse([_fake_call(f"iter-{i}", f"2026-03-{22+i:02d}")])
    for i in range(MAX_ITERATIONS)
]
_iteration_answer, _iteration_events, _iteration_requests = _run_fake_provider(
    _iteration_responses)
assert _iteration_events[-1]["stop"] == "iteration_budget"
assert "allowed steps" in _iteration_answer
_first_request = _iteration_requests[0]
assert _first_request["model"] == "gpt-5.6-luna"
assert _first_request["store"] is False
assert _first_request["parallel_tool_calls"] is False
assert _first_request["reasoning"] == {"effort": "none"}
assert _first_request["max_output_tokens"] == MAX_OUTPUT_TOKENS
assert all("function" not in tool for tool in _first_request["tools"]),     "Responses custom function tools use the flat endpoint envelope"

_success_answer, _, _success_requests = _run_fake_provider([
    _FakeResponse([_fake_call("paired-call", "2026-03-25")]),
    _FakeResponse([], "The synthetic record contains 6,781 steps."),
])
assert "6,781" in _success_answer
_second_input = _success_requests[1]["input"]
_paired_output = next(item for item in _second_input
                      if isinstance(item, dict)
                      and item.get("type") == "function_call_output")
assert _paired_output["call_id"] == "paired-call"

_provider_failure, _provider_failure_events, _provider_failure_requests =     _run_fake_provider(
        [_FakeResponse([_fake_call("failed-call", "2026-03-25")])],
        {"get_step_count": _invalid_step_result},
    )
assert _provider_failure == "The tool returned an unusable result."
assert len(_provider_failure_requests) == 1,     "A failed result must stop before the provider can generate a health answer"
assert _provider_failure_events[-1]["stop"] == "tool_error"
assert "6,781" not in _provider_failure

_many_calls = [_fake_call(f"call-{i}", f"2026-03-{22+i:02d}")
               for i in range(MAX_TOOL_CALLS + 1)]
_call_answer, _call_events, _ = _run_fake_provider([_FakeResponse(_many_calls)])
assert _call_events[-1]["stop"] == "tool_budget"
assert "too many tool requests" in _call_answer

_same_call_responses = [
    _FakeResponse([_fake_call(f"retry-{i}", "2026-03-25")])
    for i in range(MAX_RETRIES + 2)
]
_retry_answer, _retry_events, _ = _run_fake_provider(_same_call_responses)
assert _retry_events[-1]["stop"] == "retry_budget"
assert "same request was attempted" in _retry_answer

_configured_client = client
client = _FakeClient([])
try:
    run_agent("unapproved personal question", tools, tool_functions,
              verbose=False, use_openai=True)
    raise AssertionError("An unapproved provider question should be rejected")
except ValueError:
    pass
finally:
    client = _configured_client

# Timestamps are left out because their random microseconds could contain any digits.
_audit_without_times = [{k: v for k, v in event.items() if k != "timestamp"}
                        for event in AUDIT_LOG]
assert AUDIT_LOG and "6781" not in json.dumps(_audit_without_times),     "Audit events should record decisions, not health values"
print("Part 1 checks passed: boundary failures and all three budgets were exercised.")

### One full run with the real model

This cell runs the whole loop with the real model and the same controller. It
sends only the question shown, the tool schema, and the made-up result.

**Predict first:** how many model calls will it take?

In [ ]:
print("Question:", OPENAI_SYNTHETIC_QUESTION)

def _show_recorded_run():
    if RECORDED_REAL is None:
        print("No recording is available yet. Add a key to run this live.")
    else:
        print(f"Recorded run ({RECORDED_REAL['model']}, {RECORDED_REAL['recorded_on']}):")
        print("Agent:", RECORDED_REAL["final_answer"])
        print("Model calls used:", RECORDED_REAL["model_calls"])

if client is None:
    _show_recorded_run()
else:
    _before = REAL_CALLS_USED
    try:
        print("Agent:", run_agent(
            OPENAI_SYNTHETIC_QUESTION,
            tools,
            tool_functions,
            use_openai=True,
        ))
        print("Model calls used:", REAL_CALLS_USED - _before)
    except Exception as exc:
        print(f"The model call failed ({type(exc).__name__}). Showing the recording instead.")
        _show_recorded_run()

---

## Part 2: Write a heart-rate tool

*About 15 minutes.* Add a second tool, `get_heart_rate_summary`, next to the step
tool. You write the function; its schema is provided.

### Write the function

Run the TODO cell once as it is: it prints the column names of `wearable_data`.
Then replace the three `None` values, using `get_step_count` as a template.
Resting heart rate and calories each have one column; active minutes are the sum
of the two activity-minute columns. Wrap each value in `int(...)`. Leave `date`
and `unit` as they are, because the controller checks them too.

In [ ]:
print("Columns:", list(wearable_data.columns))

def get_heart_rate_summary(date: str) -> dict:
    """Return one dated summary, or an error dict when no row matches."""
    row = wearable_data[wearable_data["date"] == date]
    if row.empty:
        return {"error": f"No data for {date}."}
    return {
        "date": date,
        "resting_heart_rate_bpm": None,  # TODO: int(row.iloc[0]["..."])
        "calories_burned": None,        # TODO: int(...)
        "active_minutes": None,        # TODO: add the two activity-minute columns
        "unit": "bpm",
    }

print(get_heart_rate_summary("2026-03-25"))
print(get_heart_rate_summary("2026-01-01"))


### The schema (provided)

Read the description in this schema. It says what the tool returns and when to
use it, and it tells the model not to use this tool for plain step-count
questions. Part 3 tests how much that wording matters.

In [ ]:
# Chat Completions tool envelope (nested under `function`).
heart_rate_tool = {
    "type": "function",
    "function": {
        "name": "get_heart_rate_summary",
        "description": (
            "Get the user's resting heart rate (bpm), calories burned, and active "
            "minutes for a specific date. Use this for questions about heart rate, "
            "pulse, calories, exertion, or how hard/active the user was on a day — "
            "not for plain step-count questions."
        ),
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "date": {
                    "type": "string",
                    "description": "The date to look up, YYYY-MM-DD. Data covers 2026-03-22 to 2026-04-04.",
                }
            },
            "required": ["date"],
        },
        "strict": True,
    },
}
print("Heart-rate tool schema defined.")

### Run both tools

Register both tools and run four questions through the controller. These
questions use prepared model replies, so this cell tests your function, not the
model's choice of tool.

In [ ]:
tools_v2 = [step_count_tool, heart_rate_tool]
tool_functions_v2 = {"get_step_count": get_step_count,
                     "get_heart_rate_summary": get_heart_rate_summary}

for q in ["How many steps did I take on March 25?",
          "What was my resting heart rate on March 25?",
          "Was I active on March 27?",
          "What was my heart rate on March 28?"]:
    print(f"Q: {q}")
    print("Agent:", run_agent(q, tools_v2, tool_functions_v2))
    print("-" * 60)

### Checkpoint: Part 2

Checks your function's values and that the controller accepts its result. It
should print `Part 2 checks passed.`

In [ ]:
hr = get_heart_rate_summary("2026-03-25")
assert "resting_heart_rate_bpm" in hr, "Include a resting heart rate (bpm)"
assert hr["resting_heart_rate_bpm"] == 67, f"Expected 67 bpm on Mar 25, got {hr['resting_heart_rate_bpm']}"
assert hr["calories_burned"] == 2100, f"Expected 2100 calories, got {hr['calories_burned']}"
assert hr["active_minutes"] == 28, f"Expected 28 active minutes (18+10), got {hr['active_minutes']}"
assert "error" in get_heart_rate_summary("2026-01-01"), "Missing dates should return an error dict"
_integrated_hr = execute_tool(
    {"id": "heart-check", "name": "get_heart_rate_summary",
     "arguments": '{"date":"2026-03-25"}'},
    _registry({"get_heart_rate_summary": get_heart_rate_summary}))
assert _integrated_hr["ok"], "The controller rejected your result: include date, unit, and integer measurements."
assert _integrated_hr["data"]["date"] == "2026-03-25"
assert _integrated_hr["data"]["unit"] == "bpm"

assert heart_rate_tool["function"]["name"] == "get_heart_rate_summary", "Schema name must match the function"
assert "date" in heart_rate_tool["function"]["parameters"]["properties"], "Schema needs a 'date' parameter"
assert len(heart_rate_tool["function"]["description"]) > 20, "The description should say what the tool returns and when to use it"
print("Part 2 checks passed.")

---

## Part 3: Tool descriptions and answerability

*About 15 minutes.* A sleep tool is provided. Run it, then route four questions
with two versions of the heart-rate description. For each question, you decide
which tool, if any, has the data to answer it.

### The sleep tool (provided)

Run it. It follows the same pattern as the first two tools.

In [ ]:
def get_sleep_summary(date: str) -> dict:
    """Return a sleep summary dict for `date`, or an error dict if missing."""
    row = wearable_data[wearable_data["date"] == date]
    if row.empty:
        return {"error": f"No data for {date}. Available range: 2026-03-22 to 2026-04-04."}
    return {
        "date": date,
        "hours_asleep": round(float(row.iloc[0]["minutes_asleep"]) / 60, 1),
        "sleep_efficiency": int(row.iloc[0]["sleep_efficiency"]),
        "deep_sleep_minutes": int(row.iloc[0]["deep_sleep_min"]),
        "rem_sleep_minutes": int(row.iloc[0]["rem_sleep_min"]),
        "unit": "hours",
    }

# Chat Completions tool envelope (nested under `function`).
sleep_tool = {
    "type": "function",
    "function": {
        "name": "get_sleep_summary",
        "description": (
            "Get the user's sleep for a specific date: hours asleep, sleep "
            "efficiency, and deep/REM minutes. Use this for questions about sleep "
            "duration, sleep quality, how well the user slept, or sleep stages."
        ),
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "date": {
                    "type": "string",
                    "description": "The date to look up, YYYY-MM-DD. Data covers 2026-03-22 to 2026-04-04.",
                }
            },
            "required": ["date"],
        },
        "strict": True,
    },
}

print(get_sleep_summary("2026-03-24"))
print(get_sleep_summary("2026-01-01"))   # error dict

### All three tools together

One question per tool, then one broad question. Watch the broad one: the model
requests several tools, so you see several `[REQUEST]` lines.

In [ ]:
all_tools = [step_count_tool, heart_rate_tool, sleep_tool]
all_tool_functions = {
    "get_step_count": get_step_count,
    "get_heart_rate_summary": get_heart_rate_summary,
    "get_sleep_summary": get_sleep_summary,
}

for q in ["How many steps did I take on March 25?",
          "What was my resting heart rate on March 25?",
          "How well did I sleep on March 24?",
          "Give me a full health summary for March 28."]:   # broad -> multiple tools
    print(f"Q: {q}")
    print("Agent:", run_agent(q, all_tools, all_tool_functions))
    print("-" * 60)

### Checkpoint: sleep tool

It should print `Sleep tool checks passed.`

In [ ]:
s = get_sleep_summary("2026-03-24")
assert s["hours_asleep"] == 7.5, f"Expected 7.5 hours on Mar 24, got {s['hours_asleep']}"
assert s["sleep_efficiency"] == 95 and s["deep_sleep_minutes"] == 105 and s["rem_sleep_minutes"] == 110
assert get_sleep_summary("2026-03-26")["hours_asleep"] == 5.2, "Mar 26 should round to 5.2 hours"
assert "error" in get_sleep_summary("2026-01-01")
assert len(all_tools) == 3 and set(all_tool_functions) == {
    "get_step_count", "get_heart_rate_summary", "get_sleep_summary"}
print("Sleep tool checks passed.")

### The routing experiment

Routing means choosing which tool to call for a question. The collapsed cell
routes each question twice: once with a three-word heart-rate description
(`DESC_TERSE`) and once with a detailed one (`DESC_VERBOSE`). Nothing else changes.

The router in this cell is a simple stand-in for a model. It picks the tool
whose description shares the most words with the question. A real model also
uses meaning, and you compare the two routers below.

As you read the table, ask two separate questions about each row: which tool was
picked, and can that tool's data answer the question? A daily resting heart rate
cannot show an irregular heart rhythm.

In [ ]:
def route_offline(query, tools):
    """Deterministic stand-in for LLM routing: pick the tool whose DESCRIPTION
    shares the most words with the query (substring-aware). Returns the tool name,
    or 'none' if nothing overlaps. This exposes wording effects, not meaning
    or whether the tool can answer the question."""
    stop = {"a","an","the","of","for","to","on","in","and","or","is","are","was","were",
            "be","this","that","when","use","user","users","get","did","do","i","my","me",
            "how","about","asks","ask","their","they","them","specific","particular","day",
            "date","from","data","with","at","during","including","average","numbers",
            "recently","any","good","hard","levels","out","returns","return"}
    def toks(t):
        return [w for w in re.findall(r"[a-z]+", t.lower()) if w not in stop and len(w) >= 3]
    q = toks(query)
    best, best_score = "none", 0
    for tool in tools:
        d = set(toks(tool["function"]["description"]))
        score = sum(1 for qt in q
                    if any(qt == dt or (len(qt) >= 4 and qt in dt) or (len(dt) >= 4 and dt in qt)
                           for dt in d))
        if score > best_score:
            best, best_score = tool["function"]["name"], score
    return best


def hr_tool_with(description):
    """Nested Chat Completions schema carrying a description to test."""
    return {"type": "function", "function": {
        "name": "get_heart_rate_summary", "description": description,
        "parameters": {"type": "object", "additionalProperties": False,
                       "properties": {"date": {"type": "string"}},
                       "required": ["date"]},
        "strict": True}}

DESC_TERSE = "get HR data"
DESC_VERBOSE = ("Get the user's heart rate for a date: resting heart rate, calories burned, "
                "and active minutes. Use when the user asks about cardio, exertion, exercise "
                "intensity, workouts, or how active they were.")

ambiguous_queries = [
    "was I active yesterday",
    "how was my heart",
    "did I get good exercise",
    "my cardio numbers",
    "my energy and exertion",
    "intensity of my workouts",
    "how hard did I work out",
    "any irregular heart activity",
]

core_queries = ["was I active yesterday", "how was my heart",
                "my cardio numbers", "any irregular heart activity"]
routing_queries = ambiguous_queries if RUN_EXTRA_ROUTING else core_queries

rows = []
for query in ambiguous_queries:
    rows.append({
        "query": query,
        "terse_chose":   route_offline(query, [step_count_tool, hr_tool_with(DESC_TERSE)]),
        "verbose_chose": route_offline(query, [step_count_tool, hr_tool_with(DESC_VERBOSE)]),
    })
routing_table = pd.DataFrame(rows)
shown = routing_table[routing_table["query"].isin(routing_queries)]
print("DESC_TERSE:  ", DESC_TERSE)
print("DESC_VERBOSE:", DESC_VERBOSE)
print()
print(shown.to_string(index=False))

hr = lambda col: (shown[col] == "get_heart_rate_summary").sum()
print(f"\nHeart-rate tool picked: terse {hr('terse_chose')}/{len(shown)}, verbose {hr('verbose_chose')}/{len(shown)}")
print("Being picked more often does not make the answers right.")

### Your task: label the four questions

For each question, decide which tool has the data to answer it. Use one of
`get_step_count`, `get_heart_rate_summary`, `either`, `none`, or
`needs_clarification`. Use `none` when no tool has the data, and
`needs_clarification` when the question is too vague to route.

Run the cell and compare your labels (`intended`) with the router's choices.
Note one row where they differ. In the take-home you defend two of your labels.

Optional: set `DESC_YOURS` to a description of your own to add a third column.
Describe only what the function returns. With a key, the model comparison below
then uses your description. You can also set `RUN_EXTRA_ROUTING = True` in
Setup, rerun from there, and label all eight questions.

In [ ]:
# TODO: Replace each "TODO" with one of the allowed labels:
# get_step_count / get_heart_rate_summary / either / none / needs_clarification
MY_TARGETS = {
    "was I active yesterday": "TODO",
    "how was my heart": "TODO",
    "my cardio numbers": "TODO",
    "any irregular heart activity": "TODO",
}

# Optional: replace None with your own heart-rate description (a string).
DESC_YOURS = None

result = routing_table.copy()
if DESC_YOURS:
    result["yours_chose"] = [route_offline(q, [step_count_tool, hr_tool_with(DESC_YOURS)])
                             for q in ambiguous_queries]
result["intended"] = result["query"].map(MY_TARGETS)
print(result[result["query"].isin(routing_queries)].to_string(index=False))


### Checkpoint: routing labels

Checks that every question shown has an allowed label. It does not grade your
labels; you defend two of them in the take-home. It should print
`Routing labels recorded.`

In [ ]:
assert len(routing_table) == 8, "The terse-vs-verbose table should have 8 rows"
assert {"query", "terse_chose", "verbose_chose"}.issubset(routing_table.columns)
_allowed_targets = {"get_step_count", "get_heart_rate_summary", "either", "none", "needs_clarification"}
assert set(routing_queries).issubset(MY_TARGETS) and set(MY_TARGETS).issubset(ambiguous_queries), \
    "Label every question shown in the table"
assert all(target in _allowed_targets for target in MY_TARGETS.values()), \
    "Replace every TODO in MY_TARGETS with one of the allowed labels"
assert DESC_YOURS is None or (isinstance(DESC_YOURS, str) and len(DESC_YOURS) > 20), \
    "DESC_YOURS is optional: leave it as None or write a full description"
assert "intended" in result.columns, "Run the labeling cell above first"
print("Routing labels recorded.")

### Compare with the real model

The word-overlap router only matches words. The next cell gives the real model
the same four questions and the detailed description, then puts the two routers'
choices side by side. The model only picks a tool; no tool runs. With a key this
takes four model calls. Without one you see a recorded run.

**Predict first:** on which question will the two routers disagree?

After you run it, look at "any irregular heart activity". Can the tool that each
router picked answer it? Compare with your own label. This is a single run, and
the model may choose differently on another run.

In [ ]:
ROUTER_INSTRUCTIONS = (
    "You route questions for a personal health assistant. Today's date is 2026-04-04. "
    "If a question gives no date, use today's date. Call a tool when one fits."
)

def route_real(query, tools):
    """Return the tool the model asks for, or 'none'. The tool never runs."""
    _spend_real_call()
    response = client.responses.create(
        model=OPENAI_MODEL,
        instructions=ROUTER_INSTRUCTIONS,
        input=query,
        tools=[to_responses_tool(tool) for tool in tools],
        tool_choice="auto",
        parallel_tool_calls=False,
        reasoning={"effort": "none"},
        max_output_tokens=MAX_ROUTE_OUTPUT_TOKENS,
        store=False,
    )
    for item in response.output:
        if getattr(item, "type", None) == "function_call":
            return item.name
    return "none"

# Your own description when you wrote one; otherwise the detailed one.
ROUTE_DESCRIPTION = DESC_YOURS or DESC_VERBOSE
ROUTE_DESCRIPTION_NAME = "your DESC_YOURS" if DESC_YOURS else "DESC_VERBOSE"
ROUTE_TOOLS = [step_count_tool, hr_tool_with(ROUTE_DESCRIPTION)]

def compare_question(question):
    """Print which tool each router picks for one question of your own."""
    overlap = route_offline(question, ROUTE_TOOLS)
    if client is None:
        real = "(needs a key)"
    else:
        try:
            real = route_real(question, ROUTE_TOOLS)
        except Exception as exc:
            real = f"(call failed: {type(exc).__name__})"
    print(f"{question}\n  word overlap: {overlap} | real model: {real}")

def _real_column():
    if client is None:
        return None
    try:
        return [route_real(query, ROUTE_TOOLS) for query in routing_queries]
    except Exception as exc:
        print(f"A model call failed ({type(exc).__name__}). Using the recording instead.")
        return None

_real = _real_column()
if _real is not None:
    comparison = pd.DataFrame({"query": routing_queries,
                               "word_overlap": [route_offline(query, ROUTE_TOOLS) for query in routing_queries],
                               "real_model": _real})
    print(f"Live run: both routers used {ROUTE_DESCRIPTION_NAME}.")
elif RECORDED_REAL is not None:
    comparison = pd.DataFrame({"query": routing_queries,
                               "word_overlap": routing_table.set_index("query").loc[routing_queries, "verbose_chose"].tolist(),
                               "real_model": [RECORDED_REAL["routes"]["verbose"][q]
                                              for q in routing_queries]})
    print(f"Recorded run ({RECORDED_REAL['model']}, {RECORDED_REAL['recorded_on']}): "
          "both routers used DESC_VERBOSE.")
    if DESC_YOURS:
        print("The recording cannot test your own description; that needs a key.")
else:
    comparison = None
    print("No recording is available yet. Add a key to run this live.")

if comparison is not None:
    print(comparison.to_string(index=False))
    _differ = (comparison["word_overlap"] != comparison["real_model"]).sum()
    print(f"\nRows where the two routers disagree: {_differ}/{len(routing_queries)}")

### Optional challenge: make the two routers disagree

Write two questions of your own for which the word-overlap router and the real
model pick different tools. Use made-up questions with nothing about a real
person. Each question uses one model call. Without a key you see only the
word-overlap choice, because a new question cannot be replayed from the recording.

In [ ]:
# TODO: Write two questions that make the routers disagree. Change them and rerun.
MY_QUESTIONS = [
    "TODO: your first question",
    "TODO: your second question",
]

for question in MY_QUESTIONS:
    if "TODO" not in question:
        compare_question(question)


---

## What you built

A health agent with three read-only tools behind a controller.

**Three points to keep:**
- **A tool call is a request.** The model asks. Application code checks who is
  asking, validates the arguments, runs the tool, and records what happened.
- **Descriptions change routing.** You changed only the description and watched
  which tool was picked. A picked tool still needs the data to answer, and
  `none` or `needs_clarification` can be the right outcome.
- **The controller checks data in both directions.** It validates arguments
  before a tool runs and results after it, its error messages hide internal
  details, and call limits stop a run that keeps looping.

---

## In-class submission (attendance)

**Submit this notebook before you leave.** It counts for attendance even if a
checkpoint has not passed, so submit what you have.

Your in-class submission is Parts 1–3:
- your two requests against the controller (Part 1),
- `get_heart_rate_summary` (Part 2),
- your four labels in `MY_TARGETS` and the router comparison (Part 3).

*Colab: File → Download → Download .ipynb, then upload to the course page.*

The take-home below continues in this notebook. The reflection at the very end
is part of it.

---

# Take-home: explain the routing and repair the trend

Continue in this notebook. The required take-home has three pieces: the
trend-tool repair, a short write-up in four sections, and the two-question
reflection at the end. Aim for about 45–75 minutes. It needs no API key: for the
router comparison, use your live result if you have one, or the recorded run.

The multi-metric ranking section is an **optional extension**. Skip it unless you
want more practice designing and evaluating a decision rule. Submit the notebook
by Wednesday, November 18; check the course page for the exact time.

## Required: repair a trend tool that looks right but is wrong

A teammate added `compute_step_trend()` to report whether your walking went **up
or down** this week versus last week. It runs, it returns a clean number, and it
is **wrong**. Read it, run it, and check which dates it treats as this week and
which as last week.

In [ ]:
# As delivered by your teammate: it runs without error. Does it tell the truth?
def compute_step_trend() -> dict:
    """Compare this week's average steps (last 7 days) with last week's (first 7)."""
    recent = wearable_data.head(7)["steps"].mean()
    prior  = wearable_data.tail(7)["steps"].mean()
    pct = round((recent - prior) / prior * 100, 1)
    return {"recent_week_avg": round(recent, 1),
            "prior_week_avg":  round(prior, 1),
            "pct_change": pct,
            "direction": "up" if pct > 0 else "down"}

print("Teammate's tool says:", compute_step_trend())

### Diagnose it

Do not trust the result yet. Check it against the dates in the data:

1. Print the seven dates the function treats as `recent` and the seven it treats
   as `prior`.
2. Write down which dates should count as "this week" and which as "last week."
3. Compute the average steps for each of those two ranges yourself, and compare
   them with the values the function returned.

Fix `compute_step_trend` so its names, date windows, percentage, and direction all
agree. In the write-up, include the two date ranges that showed you where the
error was.

In [ ]:
def compute_step_trend() -> dict:
    """Compare THIS week's average steps (last 7 days) with LAST week's (first 7)."""
    recent = wearable_data.head(7)["steps"].mean()   # TODO: verify this line against the data before you trust it
    prior  = wearable_data.tail(7)["steps"].mean()   # TODO: verify this line against the data before you trust it
    pct = round((recent - prior) / prior * 100, 1)
    return {"recent_week_avg": round(recent, 1),
            "prior_week_avg":  round(prior, 1),
            "pct_change": pct,
            "direction": "up" if pct > 0 else "down"}

print("Corrected tool says:", compute_step_trend())

### Checkpoint: trend repair

Recomputes both weekly averages directly from the dates and checks that your
fixed function returns the same values. It should print
`Trend repair checks passed.`

In [ ]:
t = compute_step_trend()
_dates = pd.to_datetime(wearable_data["date"])
_latest = _dates.max()
_six_days = pd.Timedelta(6, unit="D")
_thirteen_days = pd.Timedelta(13, unit="D")
_recent_mask = _dates >= (_latest - _six_days)
_prior_mask = (_dates < (_latest - _six_days)) &               (_dates >= (_latest - _thirteen_days))
_prior_window_label = (
    f"{_dates[_prior_mask].min().date()} to {_dates[_prior_mask].max().date()}")
_recent_window_label = (
    f"{_dates[_recent_mask].min().date()} to {_dates[_recent_mask].max().date()}")
_expected_recent = round(wearable_data.loc[_recent_mask, "steps"].mean(), 1)
_expected_prior = round(wearable_data.loc[_prior_mask, "steps"].mean(), 1)
_expected_pct = round((_expected_recent - _expected_prior) / _expected_prior * 100, 1)
_expected_direction = "up" if _expected_pct > 0 else "down"
assert t["recent_week_avg"] == _expected_recent, "The recent value does not match the latest dated window."
assert t["prior_week_avg"] == _expected_prior, "The prior value does not match the preceding dated window."
assert t["direction"] == _expected_direction and t["pct_change"] == _expected_pct,     "Direction or percentage does not match the independently calculated windows."
print("Independent windows:", _prior_window_label, "then", _recent_window_label)
print("Trend repair checks passed.")

## Required: short write-up

Write 2–3 sentences per section below. Cite the rows and dates you inspected.

1. **Routing (build on your table).** In your four core Part 3 rows, which
   questions went to a different tool when the description changed, and why?
   Then defend two of your labels, including one `none` or
   `needs_clarification`. Quote the questions from those rows instead of giving
   general advice. If you wrote `DESC_YOURS`, say whether it attracted the
   questions you intended.
2. **Word-overlap router versus real model.** Use your real-model column, or the
   recorded run if you have no key. Pick one row where the two routers disagree
   and explain what the word-overlap router misses. If they agree on every row,
   explain the row that surprised you most. Describe it as a single run, because
   the model may choose differently next time.
3. **The wrong trend tool.** How did you catch it, and what would that bug have
   done to an agent asked "am I getting more active?" Explain why an agent's
   analysis tools need results that you can check. Copy both date windows that
   the trend checkpoint prints, in the form `YYYY-MM-DD to YYYY-MM-DD`.
4. **Safeguards for a tool that changes data.** Your tools only *read*. Name two
   safeguards a health agent would need before it offers a
   `set_step_goal(new_goal)` tool that *modifies* data.

In [ ]:
# TODO: Replace the scaffold with all four required write-up sections.
takehome_writeup = '''
1. Routing (build on your table): TODO
2. Word-overlap router versus real model: TODO
3. Wrong trend tool—detection and impact: TODO
4. Safeguards for a tool that changes data: TODO
'''.strip()
print(takehome_writeup)


### Checkpoint: write-up

This check confirms that all four labeled sections are present, that you quote at
least two routing questions word for word, and that you include both date windows
from the trend checkpoint. It does not grade your reasoning.

In [ ]:
assert isinstance(takehome_writeup, str) and len(takehome_writeup) >= 400
assert "TODO" not in takehome_writeup
_writeup_lower = takehome_writeup.lower()
for label in (
    "1. routing",
    "2. word-overlap router",
    "3. wrong trend tool",
    "4. safeguards",
):
    assert label in _writeup_lower, f"Missing labeled section: {label}"
_routing_rows_cited = sum(
    query.lower() in _writeup_lower for query in ambiguous_queries)
assert _routing_rows_cited >= 2,     "Cite at least two exact rows from the routing experiment."
assert _prior_window_label in takehome_writeup
assert _recent_window_label in takehome_writeup
print("Take-home written analysis is complete and tied to run evidence.")

## Optional extension: ranking and sensitivity

Skip this section for the required submission. Set `RUN_OPTIONAL_EXTENSION=True`
below to run it.

### Provided helper: rank by one metric

Each of your three tools answers about **one date**. Real questions often compare
*across* days: "which three days did I walk the most?" or "when was my resting
heart rate lowest?" Those need a **computation** tool, where code sorts the data
and returns the answer. Arithmetic like this belongs in code, not in the model's
head. Semantic search cannot do it either, because matching words does not rank
numbers.

The schema and the function are provided. Note the `enum`s in the schema, which
stop the model from inventing a metric name. Run the cells below. In the next
section you design your own ranking rule for a situation you choose.

In [ ]:
RUN_OPTIONAL_EXTENSION = False  # set True to explore the optional ranking section

In [ ]:
if RUN_OPTIONAL_EXTENSION:
    # Provided comparison schema. The enums constrain metric and order.
    rank_days_tool = {
        "type": "function",
        "function": {
            "name": "rank_days",
            "description": (
                "Rank the user's days by a single metric and return the top N. Use for "
                "cross-day comparisons and superlatives — 'which days did I walk most', "
                "'my lowest resting-heart-rate days', 'the nights I slept least'."
            ),
            "parameters": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "metric": {"type": "string",
                               "enum": ["steps", "resting_hr", "minutes_asleep",
                                        "calories_out", "sleep_efficiency"],
                               "description": "Which column to rank on."},
                    "order": {"type": "string", "enum": ["desc", "asc"],
                              "description": "desc = highest first, asc = lowest first."},
                    "n": {"type": "integer", "minimum": 1, "maximum": 14,
                          "description": "How many days to return (1-14)."},
                },
                "required": ["metric", "order", "n"],
            },
            "strict": True,
        },
    }
    print("rank_days schema ready.")
else:
    print("Optional extension skipped.")

In [ ]:
if RUN_OPTIONAL_EXTENSION:
    def rank_days(metric: str, order: str = "desc", n: int = 3) -> dict:
        """Return the top-n days ranked by `metric`. `order` is 'desc' or 'asc'.
        Return a structured dict; return an error dict for an unknown metric."""
        valid = {"steps", "resting_hr", "minutes_asleep", "calories_out", "sleep_efficiency"}
        if metric not in valid:
            return {"error": f"Unknown metric '{metric}'. Valid metrics: {sorted(valid)}."}
        ascending = (order == "asc")
        top = wearable_data.sort_values(metric, ascending=ascending).head(n)
        return {
            "metric": metric,
            "order": order,
            "days": [{"date": r["date"], metric: int(r[metric])} for _, r in top.iterrows()],
        }


    print(rank_days("steps", "desc", 3))
    print(rank_days("resting_hr", "asc", 1))
else:
    print("Optional extension skipped.")

### Checkpoint: provided helper

Confirms `rank_days` sorts correctly and handles a bad metric. Should print
`Provided rank_days helper checks passed.`

In [ ]:
if RUN_OPTIONAL_EXTENSION:
    r = rank_days("steps", "desc", 3)
    assert [d["date"] for d in r["days"]] == ["2026-03-31", "2026-03-24", "2026-03-27"], \
        f"Top-3 step days wrong: {[d['date'] for d in r['days']]}"
    assert r["days"][0]["steps"] == 13250, "Highest step day should be 13,250 (Mar 31)"
    assert rank_days("resting_hr", "asc", 1)["days"][0]["date"] == "2026-03-24", \
        "Lowest resting-HR day should be Mar 24 (58 bpm)"
    assert "error" in rank_days("heartbeat", "desc"), "An unknown metric should return an error dict"
    print("Provided rank_days helper checks passed.")
else:
    print("Optional extension skipped.")

### Provided helper: put one column on a 0–1 scale

A rule that combines several columns needs them on the same scale. This helper
is complete: it maps one numeric column to the range 0–1 (min-max
normalization). If every value in the column is the same, it returns all zeros
instead of dividing by zero. Run the example, then use the helper in `best_days`.
You still choose the columns, the signs, the weights, and what the result means.

In [ ]:
if RUN_OPTIONAL_EXTENSION:
    def min_max_normalize(series: pd.Series) -> pd.Series:
        """Map a numeric pandas Series to [0, 1]; constant input becomes zeros."""
        values = series.astype(float)
        span = values.max() - values.min()
        if span == 0:
            return pd.Series(0.0, index=values.index)
        return (values - values.min()) / span


    _normalization_demo = min_max_normalize(pd.Series([10, 20, 30]))
    assert _normalization_demo.round(2).tolist() == [0.0, 0.5, 1.0]
    assert min_max_normalize(pd.Series([7, 7])).tolist() == [0.0, 0.0]
    print("Normalization helper ready:", _normalization_demo.round(2).tolist())
else:
    print("Optional extension skipped.")

### Optional: design and evaluate your own decision rule

`rank_days` ranks by **one** column. Real decisions usually depend on several,
and no single ranking is right for everyone. Choose a situation that matters to
you (for example, picking days to review after a schedule change, or finding
balanced days to discuss with a coach), describe it in `DECISION_CONTEXT`, and
build `best_days(n)` from at least three columns. You choose the columns, the
weights, and whether a higher value should raise or lower the score. Do not use
a general clinical cutoff as your rule.

One thing to watch: the columns have very different scales (steps in the
thousands, efficiency from 0 to 100, resting heart rate in the 60s). If you add
them as they are, steps outweigh everything else, so normalize each column first.

Then answer in `decision_rule_defense`: (1) state your context and formula;
(2) explain why the rule helps with that decision, without claiming that it
measures health; (3) name one date and say how its rank differs from its rank by
steps alone; and (4) change one weight, rerun, and say which day moved the most,
or write "no rank changed" and name a date that shows it. Comparing the rules is
your evaluation.

The checkpoint only checks that every piece is filled in. Credit comes from your
own decision context, the difference you observed between the rules, and what
happened when you changed a weight.

In [ ]:
if RUN_OPTIONAL_EXTENSION:
    # TODO: Complete the decision context, two priority rules, function, and defense.
    DECISION_CONTEXT = "TODO: name the decision this ranking is meant to support"
    MY_PRIORITIES = {
        # TODO: choose at least three fields and signed weights. Positive means a
        # larger normalized value raises the score; negative means it lowers it.
        # Example shape only: "steps": 0.3
    }

    ALTERNATE_PRIORITIES = {
        # TODO: copy your keys, then change at least one signed weight.
    }

    def best_days(n: int = 3, priorities=None) -> dict:
        """Rank days for DECISION_CONTEXT using MY_PRIORITIES.
        Return {"context": ..., "formula": ...,
                "days": [{"date": ..., "score": ...}, ...]}."""
        # TODO: call min_max_normalize(df[field]) for each chosen field, apply your
        # signed weights, and rank.
        # The code supports your decision; the judgment is in the weights you choose.
        raise NotImplementedError("Design best_days before running the next cell")


    top = best_days(3, MY_PRIORITIES)
    alternate = best_days(3, ALTERNATE_PRIORITIES)
    print("Formula:", top["formula"])
    print("Decision-rule top-3:", [d["date"] for d in top["days"]])
    print("Alternate-rule top-3:", [d["date"] for d in alternate["days"]])
    print("Raw-steps top-3:", [d["date"] for d in rank_days("steps", "desc", 3)["days"]])

    # The full ordering lets you check which days changed rank between the rules.
    _all_days = len(wearable_data)
    decision_order = [d["date"] for d in best_days(_all_days, MY_PRIORITIES)["days"]]
    alternate_order = [d["date"] for d in best_days(_all_days, ALTERNATE_PRIORITIES)["days"]]
    steps_order = [d["date"] for d in rank_days("steps", "desc", _all_days)["days"]]
    rank_delta = pd.DataFrame({"date": sorted(wearable_data["date"].tolist())})
    rank_delta["decision_rank"] = rank_delta["date"].map(
        {date: rank for rank, date in enumerate(decision_order, start=1)})
    rank_delta["alternate_rank"] = rank_delta["date"].map(
        {date: rank for rank, date in enumerate(alternate_order, start=1)})
    rank_delta["steps_rank"] = rank_delta["date"].map(
        {date: rank for rank, date in enumerate(steps_order, start=1)})
    rank_delta["decision_vs_steps"] = (
        rank_delta["steps_rank"] - rank_delta["decision_rank"])
    rank_delta["alternate_vs_decision"] = (
        rank_delta["decision_rank"] - rank_delta["alternate_rank"])
    print("\nFull rank evidence:")
    print(rank_delta.sort_values("decision_rank").to_string(index=False))

    decision_rule_defense = """
    TODO: Explain your context, your formula, and what the rule does and does not
    claim. Give one exact date and how its rank differs from its rank by steps
    alone. Then give one exact date and how its rank changed under the alternate
    weights (or write "no rank changed" and name a date that shows it).
    """.strip()
    print(decision_rule_defense)
else:
    print("Optional extension skipped.")

### Checkpoint: decision rule complete

This checks that every piece is filled in, not whether your priorities are right.

In [ ]:
if RUN_OPTIONAL_EXTENSION:
    _decision = best_days(3, MY_PRIORITIES)
    _alternate = best_days(3, ALTERNATE_PRIORITIES)
    assert isinstance(DECISION_CONTEXT, str) and len(DECISION_CONTEXT) > 20 and "TODO" not in DECISION_CONTEXT

    def _valid_priorities(priorities):
        if not isinstance(priorities, dict):
            return False
        values = list(priorities.values())
        return (
            len(priorities) >= 3
            and set(priorities).issubset(wearable_data.select_dtypes(include="number").columns)
            and all(type(value) in {int, float}
                    and math.isfinite(float(value))
                    and float(value) != 0.0
                    for value in values)
        )

    assert _valid_priorities(MY_PRIORITIES),     "Use finite, nonzero numeric weights across at least three numeric fields; justify their directions."
    assert _valid_priorities(ALTERNATE_PRIORITIES)
    assert set(ALTERNATE_PRIORITIES) == set(MY_PRIORITIES)
    assert ALTERNATE_PRIORITIES != MY_PRIORITIES, "Change at least one priority for the sensitivity run."
    assert len(_decision["days"]) == 3 and all("date" in d and "score" in d for d in _decision["days"])
    assert len(_alternate["days"]) == 3
    assert all(math.isfinite(float(d["score"])) for d in _decision["days"] + _alternate["days"])
    assert _decision.get("context") == DECISION_CONTEXT
    _all_dates = set(wearable_data["date"])
    assert len(decision_order) == len(_all_dates) and set(decision_order) == _all_dates
    assert len(alternate_order) == len(_all_dates) and set(alternate_order) == _all_dates
    assert len(steps_order) == len(_all_dates) and set(steps_order) == _all_dates
    assert decision_order == [d["date"] for d in best_days(len(_all_dates), MY_PRIORITIES)["days"]]
    assert alternate_order == [d["date"] for d in best_days(len(_all_dates), ALTERNATE_PRIORITIES)["days"]]
    assert steps_order == [d["date"] for d in rank_days("steps", "desc", len(_all_dates))["days"]]
    _step_changes = rank_delta.loc[rank_delta["decision_vs_steps"] != 0, "date"].tolist()
    _priority_changes = rank_delta.loc[rank_delta["alternate_vs_decision"] != 0, "date"].tolist()
    assert isinstance(decision_rule_defense, str) and len(decision_rule_defense) >= 280        and "TODO" not in decision_rule_defense
    for _column in ("decision_vs_steps", "alternate_vs_decision"):
        _changed = rank_delta.loc[rank_delta[_column] != 0, "date"].tolist()
        if _changed:
            assert any(date in decision_rule_defense for date in _changed), "Cite a changed date for each comparison that changed."
        else:
            assert "no rank changed" in decision_rule_defense.lower(), "State 'no rank changed' and cite the zero rank deltas."
            assert any(date in decision_rule_defense for date in rank_delta["date"]), "Cite a dated row showing zero rank change."
    print("Decision rule is complete; grading focuses on your evaluation.")
else:
    print("Optional extension skipped.")

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `OFFLINE MODE` | Every cell still runs without a key; real-model cells show a recorded run. To go live, save your issued key as `OPENAI_API_KEY` in Colab Secrets, enable notebook access, and rerun Setup. |
| `Reached the limit of 40 model calls` | The notebook stops calling the model after 40 calls. Use the results you have. Restarting the runtime resets the count, so restart only if you need more calls. |
| A checkpoint fails after you edit a cell | Re-run the data cell and every cell above the checkpoint, top to bottom. The data is fixed in the notebook, so the values are the same on every run. |
| Part 2 check fails on `active_minutes` | Active minutes are split across two columns in `wearable_data`. Add the two columns, and wrap each result in `int()`. |
| Offline mode says "No listed route" | The prepared model replies cover only the lab's example questions. Use one of those. |
| Routing checkpoint fails | Replace every `"TODO"` in `MY_TARGETS` with one of the five allowed labels, spelled exactly. `DESC_YOURS` can stay `None`. |
| Decision-rule checkpoint fails | Check that both priority dictionaries use the same three or more real columns, differ in at least one weight, and that both runs return three dated scores. |
| Trend checkpoint fails | Print the dates and the average for each window. Check that `recent` holds the later seven dates and `prior` the earlier seven before you change any arithmetic. |
| A real-model cell says the call failed | The cell falls back to the recording, so you can keep going. Check that your key is saved and notebook access is on, then rerun if you want a live result. Each call has a 30-second timeout, no automatic retries, a limit on output length, and `store=False`. |

## TODO: Summary and reflection

1. What did you learn about tool routing and about what the controller is responsible for? Refer to one row from your routing table and one controller check you ran, such as an argument check, a permission check, or a call limit.
2. How did you use AI while completing this lab, and what did you verify or change yourself?


In [ ]:
# TODO: Answer both questions with details from your own routing and tool runs.
what_i_learned = ""
how_i_used_ai = ""

print("What I learned:\n", what_i_learned)
print("\nHow I used AI:\n", how_i_used_ai)
